# 06 — ML Supervisionado

Funções: `ValidacaoSuposicoesRegressao`, `plotar_curva_aprendizado`, `grid_search_manual_ridge`, `grid_search_manual_Lasso`, `relatorio_metricas`, `metodo_cotovelo`, `silhuete_metodo`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
import statsmodels.api as sm


## 1. `ValidacaoSuposicoesRegressao`

Valida as 5 suposições da regressão linear em um modelo **statsmodels.OLS**:

| Suposição | Teste |
|---|---|
| Linearidade | Rainbow Test |
| Independência dos erros | Durbin-Watson |
| Homocedasticidade | Goldfeld-Quandt |
| Normalidade dos resíduos | Shapiro-Wilk |
| Multicolinearidade | VIF |

In [ ]:
from ML.supervisionado.regressao_linear.suposicoes_regressao import ValidacaoSuposicoesRegressao

np.random.seed(42)
n = 200
X_raw = np.random.randn(n, 3)
y_raw = 3 * X_raw[:, 0] + 1.5 * X_raw[:, 1] - 0.5 * X_raw[:, 2] + np.random.normal(0, 1, n)

X_sm = sm.add_constant(X_raw)
modelo_ols = sm.OLS(y_raw, X_sm).fit()

print(modelo_ols.summary())

validacao = ValidacaoSuposicoesRegressao(modelo_ols)
print("\n=== Resultados das suposicoes ===")
for suposicao, resultado in validacao.resultados.items():
    print(f"\n{suposicao.upper()}:")
    print(resultado)

plt.show()


## 2. `plotar_curva_aprendizado`

Plota RMSE de treino e validação em função do tamanho do conjunto de treino. Curvas próximas e baixas = modelo bem calibrado. Curvas altas = underfitting. Curvas afastadas = overfitting.

In [ ]:
from ML.supervisionado.regressao_linear.avaliacoes_regressao import plotar_curva_aprendizado

np.random.seed(0)
X_lc = np.random.randn(300, 3)
y_lc = 2 * X_lc[:, 0] - X_lc[:, 1] + np.random.randn(300) * 0.5

plotar_curva_aprendizado(LinearRegression(), X_lc, y_lc)
plt.title('Curva de Aprendizado — Regressao Linear')
plt.xlabel('Tamanho do conjunto de treino')
plt.ylabel('RMSE')
plt.show()


## 3. `grid_search_manual_ridge` + `grid_search_manual_Lasso`

Busca manual do melhor `alpha` via validação cruzada (R²). Retorna o alpha com maior R² médio.

In [ ]:
from ML.supervisionado.regressao_linear.regressao_lasso_ridge.grid_search_manual import (
    grid_search_manual_ridge,
    grid_search_manual_Lasso,
)

np.random.seed(1)
X_reg = np.random.randn(300, 5)
y_reg = 2 * X_reg[:, 0] - X_reg[:, 2] + np.random.randn(300) * 0.8

alphas = [0.001, 0.01, 0.1, 1, 10, 100]

print("=== Ridge ===")
grid_search_manual_ridge(alphas, X_reg, y_reg)

print("\n=== Lasso ===")
grid_search_manual_Lasso(alphas, X_reg, y_reg)


## 4. `relatorio_metricas` — Classificação Binária

Imprime AUC, Acurácia, Recall, Precisão e Especificidade. O parâmetro `thresh` define o limiar de decisão — útil para ajustar o trade-off entre recall e precisão.

In [ ]:
from ML.supervisionado.regressao_logistica.avaliacoes_RLG.metricas_modelo import relatorio_metricas

np.random.seed(42)
X_clf = np.random.randn(400, 4)
y_clf = ((X_clf[:, 0] + X_clf[:, 1]) > 0).astype(int)

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

modelo_log = LogisticRegression()
modelo_log.fit(X_tr_c, y_tr_c)
y_proba = modelo_log.predict_proba(X_te_c)[:, 1]

print("Threshold = 0.5 (padrao)")
relatorio_metricas(y_te_c, y_proba, thresh=0.5)

print("Threshold = 0.3  (aumenta recall, reduz precisao)")
relatorio_metricas(y_te_c, y_proba, thresh=0.3)


## 5. KMeans — `metodo_cotovelo` + `silhuete_metodo`

Dois auxiliares para escolha do **k ideal**:

- **Cotovelo**: procure o 'joelho' na curva de SSE  
- **Silhueta**: maior score = clusters mais bem separados

In [ ]:
from ML.N_supervisionado.kmeans import metodo_cotovelo, silhuete_metodo

np.random.seed(0)
c1 = np.random.randn(60, 2) + [0,  0]
c2 = np.random.randn(60, 2) + [6,  6]
c3 = np.random.randn(60, 2) + [0, 10]
X_km = pd.DataFrame(np.vstack([c1, c2, c3]), columns=['x', 'y'])

print("=== Metodo do Cotovelo (procure o 'joelho' em k=3) ===")
metodo_cotovelo(X_km)
plt.show()

print("\n=== Metodo da Silhueta (maior score em k=3) ===")
silhuete_metodo(X_km)
plt.show()
